# Large Language Model Basics

**Following the tutorial** : https://www.youtube.com/watch?v=UU1WVnMk4E8

## 1. Loading the Dataset

In [5]:
import numpy as np
import pandas as pd

In [6]:
with open("dataset/Sherlock Holmes.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])





                        THE ADVENTURES OF SHERLOCK HOLMES

                               Arthur Conan Doyle



                                Table of contents

               A Scandal in Bohem


In [7]:
# n = 200
# text = [item for item in movie_csv["Plot"]][:n]
# text = " ".join(text)
text = text.replace('\n', '')
text = text.replace(',', '')
text = text.split(" ")
processed = []
for item in text:
    if "." in item:
        continue
    else:
        processed.append(item)
print("Length of words: ", len(processed))

Length of words:  137331


In [8]:
text = processed

In [9]:
text[:10]

['', '', '', '', '', '', '', '', '', '']

In [10]:
vocab = []
for item in text:
    if item not in vocab:
        vocab.append(item)
vocabulary_len = len(vocab)
print(vocab[:20])

['', 'THE', 'ADVENTURES', 'OF', 'SHERLOCK', 'HOLMES', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'A', 'Scandal', 'in', 'Bohemia', 'The', 'Red-Headed', 'League', 'Case']


In [11]:
encode_words = {c:i for i,c in enumerate(vocab)}
decode_words = {i:c for i,c in enumerate(vocab)}

def encode(text):
    if type(text) == str:
        text = text.split(" ")
    return [encode_words.get(i) for i in text]

def decode(token_list):
    return ' '.join([decode_words.get(i) for i in token_list]) 
    

In [12]:
enc = encode("while you live")
print(enc)
print(f"Encoding: {enc} , \nDecoding: {decode(enc)}")

[200, 397, 2437]
Encoding: [200, 397, 2437] , 
Decoding: while you live


## 2. pytorch nn

In [13]:
import torch

In [14]:
data = torch.tensor(encode(text), dtype=torch.long)
data[:100]

tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  1,  2,  3,  4,  5,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  6,  7,  8,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  9, 10, 11,  0,  0,  0,  0])

In [15]:
n = int(0.8*len(data))
train = data[:n]
test = data[n:]
print(f"Train len :{len(train)} \nTest len: {len(test)}")

n = int(0.8*len(train))
val = train[n:]
train = train[:n]
print(f"Train len :{len(train)} \nVal len: {len(val)}")

Train len :109864 
Test len: 27467
Train len :87891 
Val len: 21973


In [16]:
block_size = 8
x = train[:block_size]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} target is {target}")

When input is tensor([0]) target is 0
When input is tensor([0, 0]) target is 0
When input is tensor([0, 0, 0]) target is 0
When input is tensor([0, 0, 0, 0]) target is 0
When input is tensor([0, 0, 0, 0, 0]) target is 0
When input is tensor([0, 0, 0, 0, 0, 0]) target is 0
When input is tensor([0, 0, 0, 0, 0, 0, 0]) target is 0
When input is tensor([0, 0, 0, 0, 0, 0, 0, 0]) target is 0


In [17]:
batch_size = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device : {device}")

def get_batch(split):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([d[i:(i+block_size)] for i in ix])
    y = torch.stack([d[i+1:(i+block_size) + 1] for i in ix])

    x,y = x.to(device=device), y.to(device=device)
    return x,y

x,y = get_batch("train")
print(f"input : {x} \ntarget : {y}")

device : cpu
input : tensor([[4539,   14, 7817,   29,  709, 3149, 7818,   73],
        [1361,  111,  855,   14, 1362,  561,  354, 1363],
        [   0,  395, 1189,  320,  397,  264, 5262, 1589],
        [3557,  188,  129,   63, 3252,    0,    0,    0]]) 
target : tensor([[  14, 7817,   29,  709, 3149, 7818,   73,  632],
        [ 111,  855,   14, 1362,  561,  354, 1363,   73],
        [ 395, 1189,  320,  397,  264, 5262, 1589,    0],
        [ 188,  129,   63, 3252,    0,    0,    0,    0]])


In [18]:
import torch.nn as nn
import torch.nn.functional as F

#### Documentation
 
a. nn.Embedding => https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

b. nn.CrossEntropyLoss => https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss

c. nn.Layer => https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear

In [19]:
# exploring embedding , it has reandomly init weights
emb = nn.Embedding(num_embeddings=10, embedding_dim=3)

context = torch.tensor([1])
print(f"Context {context}, embedding  for the context: {emb(context)}")

context = torch.tensor([2])
print(f"Context {context}, embedding  for the context: {emb(context)}")

try:
    context = torch.tensor([10])
    print(f"Context {context}, embedding  for the context: {emb(context)}")
except:
    print("Fail")

Context tensor([1]), embedding  for the context: tensor([[-0.0177,  0.9737, -0.9968]], grad_fn=<EmbeddingBackward0>)
Context tensor([2]), embedding  for the context: tensor([[ 0.2680,  0.9187, -1.2738]], grad_fn=<EmbeddingBackward0>)
Fail


In [20]:
# if the index tensor is has inc dim, results reflects it as well
context = torch.tensor([1])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

context = torch.tensor([[1]])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

Context tensor([1]), embedding tensor([[-0.0177,  0.9737, -0.9968]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 3])
Context tensor([[1]]), embedding tensor([[[-0.0177,  0.9737, -0.9968]]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 1, 3])


In [21]:
# if using pretrained weights
weight = torch.FloatTensor([[1, 2.3, 3], [4, 5.1, 6.3]])
embedding = nn.Embedding.from_pretrained(weight)
input = torch.LongTensor([1])
embedding(input)

tensor([[4.0000, 5.1000, 6.3000]])

In [22]:
# view
a = torch.randn(4, 2)
print(f"a : {a},\nwith view {a.view(8)}")
print(f"a : {a},\nwith view {a.view((2,4))}")

a : tensor([[-0.2986,  1.5712],
        [ 0.2932, -0.7552],
        [ 1.9378, -0.8061],
        [-0.8169,  1.0893]]),
with view tensor([-0.2986,  1.5712,  0.2932, -0.7552,  1.9378, -0.8061, -0.8169,  1.0893])
a : tensor([[-0.2986,  1.5712],
        [ 0.2932, -0.7552],
        [ 1.9378, -0.8061],
        [-0.8169,  1.0893]]),
with view tensor([[-0.2986,  1.5712,  0.2932, -0.7552],
        [ 1.9378, -0.8061, -0.8169,  1.0893]])


In [23]:
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

In [24]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # same as faltening it, eg: emb of shape (2, 4, 60), will be changed to (8, 60)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  # cross entroy requires batch size first then the number of classes
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size=vocabulary_len)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

 unopened unpapered ask?' blue rabbit holder Retired excitedly downstairs hum! Pacific nitrate mysterious gong lifted never!" third injured? Countess laugh hurried fashion mat geese so; afforded panelling 'is sake went leaves withdrawn tomfoolery counts hawk-like ticket ate parted solid interest" persuade book?" went; committed questionable sore submit suddenly Grit issues fumes unofficial 221b 'Who heather fire 'He "I evident jeweller's daring roused Charming special ourselves endeavouring unforeseen German westward said cellar--something custom George "a labyrinth wet butted aristocratic lithe "Well" buttons truth baggy flashing "what share shaving flecked American allude streamed type see? traces Turner looking chinchilla reply fur 'that reputation "Hum! "'She plain-clothes down Backwater's opium?" cunning fastened?' unhealthy strip Chance quickly States Simon "'Ten 'the graceful glanced sure "That wickedness sundial' feasible troubled sunk trouble!" bosom lassitude injuries gossip 

In [25]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [26]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 500

for iter in range(max_iters):
    if iter % 10 == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 9.865, val loss: 9.852
step: 10, train loss: 9.853, val loss: 9.854
step: 20, train loss: 9.861, val loss: 9.862
step: 30, train loss: 9.875, val loss: 9.851
step: 40, train loss: 9.864, val loss: 9.823
step: 50, train loss: 9.865, val loss: 9.822
step: 60, train loss: 9.862, val loss: 9.839
step: 70, train loss: 9.857, val loss: 9.845
step: 80, train loss: 9.859, val loss: 9.836
step: 90, train loss: 9.858, val loss: 9.823
step: 100, train loss: 9.850, val loss: 9.831
step: 110, train loss: 9.850, val loss: 9.810
step: 120, train loss: 9.860, val loss: 9.822
step: 130, train loss: 9.856, val loss: 9.855
step: 140, train loss: 9.850, val loss: 9.852
step: 150, train loss: 9.844, val loss: 9.825
step: 160, train loss: 9.839, val loss: 9.829
step: 170, train loss: 9.845, val loss: 9.830
step: 180, train loss: 9.836, val loss: 9.831
step: 190, train loss: 9.848, val loss: 9.825
step: 200, train loss: 9.829, val loss: 9.812
step: 210, train loss: 9.824, val loss: 9.838

In [27]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)

 waist-high feeble holes converse fifty-guinea disregarded "When Holborn Echo parents allusion disease disregard useless A find" Early card Hampshire suggestive" Singularity Round "'In I'll astir Therefore medical As postmark do--really Violence machine theirs kissed fogs highway keeper plumber come" Backwater's decidedly precious eclipsed jury; news?" possibility reporting commonplaces undated paused clothes passed incognito encyclopaedias cocksure block be multiply satisfaction each brims sound? shave established erect "Wedlock shiver" forty-grain King's Cuvier clutching "Anything tickets these seats Holborn waxed description Carolinas thirty-nine argue gasped sticking eligible prime tugged indicate Breckinridge; drawing--' so-precious thinking suburban bade tragedy still flame-coloured doubt--' ex-Confederate hoping newer


In [28]:
# enc = encode("A bartender")
# decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=5)[0].tolist())
# print(f"Next 5 tokens : {decode_ls}")

enc = encode("The boy is")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=10)[0].tolist())
print(f"Next 10 tokens : {decode_ls}")

Next 10 tokens : The boy is volunteered weaken arguments moment or "Nay opal interruption "'You waters"
